# Malinois MPRA Activity Scoring

![Malinois MPRA Activity Scoring](https://proto-bio.github.io/proto-assets/images/tool/malinois/hero.png)

Malinois is the regulatory-sequence convolutional neural network at the core of CODA (Computational Optimization of DNA Activity), published by Gosai et al. (2024). It predicts massively parallel reporter assay (MPRA) activity for 200 bp DNA inserts in three cell contexts — K562, HepG2, and SK-N-SH — and was trained on aggregated log2 fold-change measurements to enable machine-guided design of cell-type-targeting cis-regulatory elements.

This notebook demonstrates both Malinois tools: `run_malinois_score`, which returns reverse-complement-averaged raw activity for candidate inserts, and `run_malinois_gradient`, which backpropagates a weighted activity objective through relaxed `B x L x 4` DNA logits for Fast SeqProp-style sequence design.

In [1]:
from proto_tools.utils.notebook_docs import (
    display_overview,
    display_api_reference,
    display_docs_section,
    display_doc_link,
    display_available_tools,
)

display_doc_link("malinois")
display_overview("malinois")
display_docs_section("malinois", "Background")

# Malinois

Malinois is the CODA/BODA2 convolutional neural network for predicting MPRA-measured regulatory DNA activity from 200 bp inserts. This toolkit scores sequences in K562, HepG2, and SK-N-SH cell contexts and exposes a differentiable activity-gradient call for sequence design.

Malinois is the regulatory sequence model used in CODA ([Gosai et al., 2024](https://doi.org/10.1038/s41586-024-08070-z)) for machine-guided design of cell-type-targeting cis-regulatory elements. The model adapts the Basset-style convolutional architecture to MPRA data and predicts activity from a fixed 200 nucleotide insert after adding the assay flanks expected by the published checkpoint.

The model returns one raw activity value for each supported cell context: K562, HepG2, and SK-N-SH. The scoring wrapper averages forward and reverse-complement predictions and returns selected raw outputs. The gradient wrapper applies max/min sigmoid objective terms to these raw scores and backpropagates through relaxed A,C,G,T logits, matching the Fast SeqProp-style design path used for regulatory DNA optimization.

## Available tools

In [2]:
display_available_tools("malinois")

- **`run_malinois_gradient()`** — Compute differentiable Malinois MPRA activity losses and gradients for relaxed DNA logits
- **`run_malinois_score()`** — Score regulatory DNA activity using the Malinois MPRA model

### `run_malinois_score`

Given one or more 200 bp DNA inserts, this function adds the MPRA assay flanks expected by the published checkpoint, runs the forward and reverse-complement sequences through Malinois, and returns the averaged raw activity for each requested cell type. It is the tool to use when ranking or screening regulatory DNA candidates by predicted K562, HepG2, or SK-N-SH activity. Scoring is deterministic and batched, so `batch_size` changes throughput without changing results.

In [3]:
from proto_tools.tools.sequence_scoring.malinois import (
    MalinoisScoreConfig,
    MalinoisScoreInput,
    run_malinois_score,
)

In [4]:
# Display input docs
display_api_reference("malinois", "input", "run_malinois_score")

# Two real 200 bp MPRA inserts from Gosai et al. 2024 (Table S2): a strongly
# active enhancer (high measured activity in all three cell types) and a
# near-silent element (measured activity near zero).
active_enhancer = (
    "CCCACCGGGAGCCTCGGAAGGACCGGCCTCCCCTCACTCTAAGCATGCGCGTTAAAGCGAC"
    "AGCATAGGCTATAGAAAGCTTTATGAGAATAGCTGGGAGCCATACAGCCGCAGGGCCGCGT"
    "GGCCTAATGGATAAGGCGTCTGATTCCGGATCAGAAGATTGAGGGTTCGAGTCCCTTCGTG"
    "GTCGTTGCCATGTTAAC"
)
inactive_element = (
    "CATAAACAATTGCTATTCACAAATTATTCGAATCAAATCAGGTCTCAGCAAATTCCCAACT"
    "CTGCAATGTCATTGCAATGTTATATATAAAAGAGATCCTGAAATAAGTAAGCTATTATCAT"
    "ATATTTGAAAATTCGGCCAGGCATGATAGCATGCGCCTGTTATCCCAGCTACTCAGGCTGG"
    "GAAGGCTGAAGTGAGAG"
)
inputs = MalinoisScoreInput(sequences=[active_enhancer, inactive_element])

**Input** — `MalinoisScoreInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | DNA insert sequence(s) to score with Malinois |

In [5]:
# Display config docs
display_api_reference("malinois", "config", "run_malinois_score")

config = MalinoisScoreConfig(
    cell_types=["K562", "HepG2", "SKNSH"],
    batch_size=2,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `MalinoisScoreConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run Malinois inference on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>cell_types</code> | <code>list[Literal['K562', 'HepG2', 'SKNSH']]</code> | <code>['K562', 'HepG2', 'SKNSH']</code> | Malinois cell-type outputs to return. |
| <code>seq_length</code> | <code>int</code> | <code>200</code> | Expected DNA insert length before MPRA flank padding. |
| <code>artifact_path</code> | <code>str</code> | <code>''</code> | Optional local artifact tarball path; empty uses the managed cache download. |
| <code>artifact_url</code> | <code>str</code> | <code>'https://zenodo.org/records/10698014/files/MODELS-malinois_artifacts__20211113_021200__287348.tar.gz?download=1'</code> | HTTPS URL used to provision the Malinois artifact. |
| <code>artifact_md5</code> | <code>str</code> | <code>'375142a714e7df73c463b46113a65210'</code> | Expected MD5 checksum for the downloaded Malinois artifact. |
| <code>malinois_dir</code> | <code>str</code> | <code>''</code> | Optional local Malinois metadata directory; empty uses the managed cache extraction. |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Number of sequences to score simultaneously on GPU. |

In [6]:
result = run_malinois_score(inputs, config)

# Malinois recovers the measured contrast: high predicted activity for the
# active enhancer, near-zero for the inactive element.
labels = ["active_enhancer", "inactive_element"]
[{"insert": labels[idx], **item.scores} for idx, item in enumerate(result.results)]

Running run_malinois_score [00:00]

[{'insert': 'active_enhancer',
  'K562': 6.42901611328125,
  'HepG2': 4.607794761657715,
  'SKNSH': 5.777548789978027},
 {'insert': 'inactive_element',
  'K562': 0.40050989389419556,
  'HepG2': 0.1743352711200714,
  'SKNSH': 0.15775281190872192}]

### `run_malinois_gradient`

For gradient-based DNA design, this function takes batched relaxed logits of shape `(B, L, 4)` in `A,C,G,T` order, relaxes them to nucleotide probabilities at the given softmax temperature, and computes a weighted sum of per-cell sigmoid objective terms. Each term either maximizes or minimizes activity in a target cell type, so a single call expresses an on-target / off-target trade-off. With `compute_gradient=True` it backpropagates the objective to a gradient with the same shape as the input logits, ready to drive an optimizer loop.

In [7]:
import numpy as np

from proto_tools.tools.sequence_scoring.malinois import (
    MalinoisGradientConfig,
    MalinoisGradientInput,
    MalinoisGradientLossTerm,
    run_malinois_gradient,
)

In [8]:
# Display input docs
display_api_reference("malinois", "input", "run_malinois_gradient")

# A batch of two random relaxed 200 bp candidates (B, L, 4) in A,C,G,T order
rng = np.random.default_rng(0)
logits = rng.normal(size=(2, 200, 4)).tolist()
gradient_inputs = MalinoisGradientInput(logits=logits, temperature=1.0)

**Input** — `MalinoisGradientInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>logits</code> | <code>list[list[list[float]]]</code> | required | Batched relaxed DNA logits with shape (B, L, 4) in A,C,G,T order. |
| <code>temperature</code> | <code>float</code> | <code>1.0</code> | Softmax temperature used to convert logits into relaxed nucleotide probabilities. |

In [9]:
# Display config docs
display_api_reference("malinois", "config", "run_malinois_gradient")

# Maximize K562 activity while minimizing HepG2 activity
gradient_config = MalinoisGradientConfig(
    loss_terms=[
        MalinoisGradientLossTerm(cell_type="K562", direction="max"),
        MalinoisGradientLossTerm(cell_type="HepG2", direction="min"),
    ],
    compute_gradient=True,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `MalinoisGradientConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run Malinois inference on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>loss_terms</code> | <code>list[MalinoisGradientLossTerm]</code> | <code>[MalinoisGradientLossTerm(cell_type='K562', direction='max', weight=1.0, sigmoid_center=4.0, sigmoid_scale=1.0)]</code> | Per-cell Malinois objectives to sum before backpropagation. |
| <code>seq_length</code> | <code>int</code> | <code>200</code> | Expected DNA insert length before MPRA flank padding. |
| <code>artifact_path</code> | <code>str</code> | <code>''</code> | Optional local artifact tarball path; empty uses the managed cache download. |
| <code>artifact_url</code> | <code>str</code> | <code>'https://zenodo.org/records/10698014/files/MODELS-malinois_artifacts__20211113_021200__287348.tar.gz?download=1'</code> | HTTPS URL used to provision the Malinois artifact. |
| <code>artifact_md5</code> | <code>str</code> | <code>'375142a714e7df73c463b46113a65210'</code> | Expected MD5 checksum for the downloaded Malinois artifact. |
| <code>malinois_dir</code> | <code>str</code> | <code>''</code> | Optional local Malinois metadata directory; empty uses the managed cache extraction. |
| <code>soft</code> | <code>float</code> | <code>1.0</code> | Blend hard argmax one-hot (0) to softmax probabilities (1). |
| <code>hard</code> | <code>float</code> | <code>0.0</code> | Straight-through hard-forward coefficient. |
| <code>compute_gradient</code> | <code>bool</code> | <code>True</code> | Run backward pass and return gradient; set False for forward-only scoring. |

In [10]:
gradient_result = run_malinois_gradient(gradient_inputs, gradient_config)
{
    "loss": gradient_result.loss,
    "gradient_shape": np.array(gradient_result.gradient).shape,
    "raw_scores": gradient_result.metrics["raw_scores"],
}

Running run_malinois_gradient [00:00]

{'loss': 1.9998371005058289,
 'gradient_shape': (2, 200, 4),
 'raw_scores': [{'K562': 0.5882349014282227, 'HepG2': 0.5870180130004883},
  {'K562': 0.5783790349960327, 'HepG2': 0.5742796659469604}]}